# Advanced Routers
Explore LLM-as-a-Judge, Cascading, and SFT Router frameworks.


In [ ]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Cascading Router Logic
Predict reward scores using Ridge Regression and fallback to a cheaper model if the margin is below a threshold.


In [ ]:
# === CONFIGURATIONS ===
CASCADING_CONFIG = {
    "feature_config": "dense_Qwen-Qwen3-Embedding-8B",
    "fallback_model": "Model_K",  # Model to fall back to when margin is small
    "k_folds": 10,
    "random_state": 42,
    "ridge_alpha": 1.0,
    "min_theta": 0.0,
    "max_theta": 0.20,
    "num_theta_steps": 100
}


In [ ]:
# Cascading Router Logic
train = pd.read_csv(f"{DATA_DIR}/train.csv")
embeddings = np.load(f"{CACHE_DIR}/{CASCADING_CONFIG['feature_config']}_features.npy")

models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]
max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()

# Calculate true rewards
rewards = pd.DataFrame(index=train.index, columns=models)
for m in models:
    p = train[f"{m}_performance"]
    c = train[f"{m}_cost"]
    rewards[m] = 0.85 * p - 0.15 * (c / global_avg_max_cost)
y_reg = rewards.values

# Train Ridge Regressors to get out-of-fold predicted rewards
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

kf = KFold(n_splits=CASCADING_CONFIG["k_folds"], shuffle=True, random_state=CASCADING_CONFIG["random_state"])
oof_pred_rewards = np.zeros((len(train), len(models)))

print("Training Ridge OOF predictors...")
for train_idx, val_idx in kf.split(embeddings):
    X_train, X_val = embeddings[train_idx], embeddings[val_idx]
    for m_idx in range(len(models)):
        y_train = y_reg[train_idx, m_idx]
        reg = Ridge(alpha=CASCADING_CONFIG["ridge_alpha"])
        reg.fit(X_train, y_train)
        oof_pred_rewards[val_idx, m_idx] = reg.predict(X_val)

# Apply Margin Gating fallbacks & find the optimal threshold (theta)
cheap_model_idx = models.index(CASCADING_CONFIG["fallback_model"])

best_cv_reward = 0
best_theta = 0
best_preds = None

print("Tuning margin threshold (theta)...")
for theta in np.linspace(CASCADING_CONFIG["min_theta"], CASCADING_CONFIG["max_theta"], CASCADING_CONFIG["num_theta_steps"]):
    oof_preds = []
    for i in range(len(train)):
        pred_rewards = oof_pred_rewards[i]
        best_idx = np.argmax(pred_rewards)
        
        # Calculate predicted reward margin of the best model over the cheap fallback model
        margin = pred_rewards[best_idx] - pred_rewards[cheap_model_idx]
        
        # If the reward gain is too small, fallback to the cheap model
        if margin < theta:
            oof_preds.append(CASCADING_CONFIG["fallback_model"])
        else:
            oof_preds.append(models[best_idx])
            
    # Calculate CV score for this theta configuration
    pred_p = [train.loc[idx, f"{m}_performance"] for idx, m in enumerate(oof_preds)]
    pred_c = [train.loc[idx, f"{m}_cost"] for idx, m in enumerate(oof_preds)]
    avg_p = np.mean(pred_p)
    avg_c = np.mean(pred_c)
    cv_reward = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)
    
    if cv_reward > best_cv_reward:
        best_cv_reward = cv_reward
        best_theta = theta
        best_preds = oof_preds

print(f"\n[+] Optimal Margin Threshold (theta): {best_theta:.4f}")
print(f"[+] Margin-Gated CV Reward: {best_cv_reward:.5f}")
print(f"[+] Average Performance: {np.mean([train.loc[i, f'{m}_performance'] for i, m in enumerate(best_preds)]):.4f}")
print(f"[+] Average Cost: {np.mean([train.loc[i, f'{m}_cost'] for i, m in enumerate(best_preds)]):.4f}")


Training Ridge OOF predictors...
Tuning margin threshold (theta)...

[+] Optimal Margin Threshold (theta): 0.0626
[+] Margin-Gated CV Reward: 0.46692
[+] Average Performance: 0.5719
[+] Average Cost: 0.0099


In [ ]:
# Log Cascading Results to experiment_summary.csv
summary = pd.read_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv")
new_row = {
    'Experiment_ID': f"Cascading_Ridge_{CASCADING_CONFIG['fallback_model']}_theta{round(best_theta, 4)}",
    'Method_Category': 'Cascading',
    'Features': CASCADING_CONFIG['feature_config'],
    'K-Fold': CASCADING_CONFIG['k_folds'],
    'CV_Reward_0.85': round(best_cv_reward, 4),
    'CV_Avg_Performance': round(np.mean([train.loc[i, f'{m}_performance'] for i, m in enumerate(best_preds)]), 4),
    'CV_Avg_Cost': round(np.mean([train.loc[i, f'{m}_cost'] for i, m in enumerate(best_preds)]), 4),
    'Model_Distribution': str(pd.Series(best_preds).value_counts().to_dict()),
    'Public_Kaggle_Score': None
}
summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
summary.to_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv", index=False)
print("Logged Cascading experiment to experiment_summary.csv")


Logged Cascading experiment to experiment_summary.csv


/tmp/ipykernel_2271501/3943329846.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)


## 2. SFT Router / Sequence Classification Logic
Fine-tune a sequence classification transformer model to predict the best routing model.


In [ ]:
# === CONFIGURATIONS ===
SFT_CONFIG = {
    "model_name": "distilbert-base-uncased",  # Small, fast model for classification SFT
    "num_epochs": 1,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "k_folds": 5,
    "random_state": 42
}


In [ ]:
# SFT Classifier Logic
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import KFold
import gc

train = pd.read_csv(f"{DATA_DIR}/train.csv")
models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]
max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()

# Define targets (the model that maximizes the reward for each query)
rewards = pd.DataFrame(index=train.index, columns=models)
for m in models:
    p = train[f"{m}_performance"]
    c = train[f"{m}_cost"]
    rewards[m] = 0.85 * p - 0.15 * (c / global_avg_max_cost)

train['best_model_idx'] = rewards.values.argmax(axis=1)

# Dataset representation for huggingface
class TextClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 5-Fold CV
kf = KFold(n_splits=SFT_CONFIG["k_folds"], shuffle=True, random_state=SFT_CONFIG["random_state"])
oof_preds = np.zeros(len(train))
tokenizer = AutoTokenizer.from_pretrained(SFT_CONFIG["model_name"])

print("Starting SFT Sequence Classifier Fine-tuning...")
for fold, (train_idx, val_idx) in enumerate(kf.split(train)):
    print(f"\n--- Training Fold {fold+1}/{SFT_CONFIG['k_folds']} ---")
    
    # Force VRAM cleanup
    torch.cuda.empty_cache()
    gc.collect()
    
    train_queries = train.iloc[train_idx]['query'].tolist()
    val_queries = train.iloc[val_idx]['query'].tolist()
    
    train_labels = train.iloc[train_idx]['best_model_idx'].tolist()
    val_labels = train.iloc[val_idx]['best_model_idx'].tolist()
    
    train_encodings = tokenizer(train_queries, truncation=True, padding=True, max_length=128)
    val_encodings = tokenizer(val_queries, truncation=True, padding=True, max_length=128)
    
    train_dataset = TextClassificationDataset(train_encodings, train_labels)
    val_dataset = TextClassificationDataset(val_encodings, val_labels)
    
    model = AutoModelForSequenceClassification.from_pretrained(SFT_CONFIG["model_name"], num_labels=11)
    
    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        num_train_epochs=SFT_CONFIG["num_epochs"],
        per_device_train_batch_size=SFT_CONFIG["batch_size"],
        per_device_eval_batch_size=SFT_CONFIG["batch_size"],
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=100,
        eval_strategy="no",
        learning_rate=SFT_CONFIG["learning_rate"],
        fp16=torch.cuda.is_available(),
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()
    
    # Predict on validation fold
    val_predictions = trainer.predict(val_dataset)
    oof_preds[val_idx] = np.argmax(val_predictions.predictions, axis=1)
    
    # Cleanup to save VRAM
    del model, trainer
    import shutil
    if os.path.exists(f"./results_fold_{fold}"):
        shutil.rmtree(f"./results_fold_{fold}")

# Calculate CV Metrics
train['oof_pred_model_sft'] = [models[int(p)] for p in oof_preds]

pred_p = []
pred_c = []
for i, row in train.iterrows():
    pred_m = row['oof_pred_model_sft']
    pred_p.append(row[f"{pred_m}_performance"])
    pred_c.append(row[f"{pred_m}_cost"])

avg_p = np.mean(pred_p)
avg_c = np.mean(pred_c)
cv_reward = 0.85 * avg_p - 0.15 * (avg_c / global_avg_max_cost)

print("\n[+] SFT Classifier CV Reward 0.85:", round(cv_reward, 4))
print("[+] SFT Classifier Average Performance:", round(avg_p, 4))
print("[+] SFT Classifier Average Cost:", round(avg_c, 4))


In [ ]:
# Log SFT Results to experiment_summary.csv
summary = pd.read_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv")
new_row = {
    'Experiment_ID': f"SFT_{SFT_CONFIG['model_name'].replace('/', '-')}",
    'Method_Category': 'SFT_Classifier',
    'Features': 'text_tokens',
    'K-Fold': SFT_CONFIG['k_folds'],
    'CV_Reward_0.85': round(cv_reward, 4),
    'CV_Avg_Performance': round(avg_p, 4),
    'CV_Avg_Cost': round(avg_c, 4),
    'Model_Distribution': str(train['oof_pred_model_sft'].value_counts().to_dict()),
    'Public_Kaggle_Score': None
}
summary = pd.concat([summary, pd.DataFrame([new_row])], ignore_index=True)
summary.to_csv(f"{ARTIFACTS_DIR}/experiment_summary.csv", index=False)
print("Logged SFT experiment to experiment_summary.csv")
